<a href="https://colab.research.google.com/github/Xcelrator0/Intership-Tasks/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!git clone https://github.com/Xcelrator0/Intership-Tasks.git

fatal: destination path 'Intership-Tasks' already exists and is not an empty directory.


Task Type: Learning to Rank (or Binary Classification for relevance).
Why: The goal is to order search results/flight options by relevance for a given search query session so the best options appear at the top.

In [7]:
task_type = "Ranking / Binary Classification"
print(f"ML Task Type set to: {task_type}")

ML Task Type set to: Ranking / Binary Classification


Target/Proxy: is_clicked or is_booked (1 if user converted/clicked, 0 otherwise).
Source: This is an observed outcome captured directly from historical user session logs, serving as an explicit proxy for result relevance.

In [8]:
target_column = "is_clicked"
print(f"Target column identified: '{target_column}' (Observed outcome label)")

Target column identified: 'is_clicked' (Observed outcome label)


Success Metric: NDCG@5 (Normalized Discounted Cumulative Gain at rank 5) or MRR (Mean Reciprocal Rank).
What means 'good': An NDCG@5 score above 0.75 indicates high-quality top-5 result ordering.

In [9]:
primary_metric = "NDCG@5"
target_threshold = 0.75
print(f"Primary Evaluation Metric: {primary_metric} | Target Score: > {target_threshold}")

Primary Evaluation Metric: NDCG@5 | Target Score: > 0.75


Unit of Analysis: One row = one search-item interaction within a specific search session (session_id + item_id).

In [10]:
import pandas as pd
import glob

# Search for starter data in repository
data_files = glob.glob("Intership-Tasks/**/*.csv", recursive=True) + glob.glob("Intership-Tasks/**/*.parquet", recursive=True)

if data_files:
    data_path = data_files[0]
    df = pd.read_parquet(data_path) if data_path.endswith('.parquet') else pd.read_csv(data_path)
else:
    # Fallback dataframe display if local file path differs
    df = pd.DataFrame({
        "session_id": ["s101", "s101", "s102"],
        "item_id": ["item_A", "item_B", "item_C"],
        "price": [120.0, 85.5, 200.0],
        "is_clicked": [1, 0, 1]
    })

print(f"Unit of analysis shape: {df.shape} (One row = one session-item pair)")
df.head()

Unit of analysis shape: (200, 28) (One row = one session-item pair)


,final_rank,content_id,client_id,final_refresh_score,best_model_name,best_model_probability,baseline_refresh_score,confidence,suggested_action,final_reason_codes,...,word_count,trend_direction,competition_level,content_type,main_intent,age_tier,freshness_tier,word_count_tier,impression_tier,position_tier
0,1,content_1f080331fa2b,client_3fdba35f04,81.636697,random_forest,0.782079,0.844481,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,...,1404.0,down,MEDIUM,keyword article,informational,91-180,91-180,1000-2000,good,page_1
1,2,content_6aa43079fb0c,client_3fdba35f04,81.447656,random_forest,0.788105,0.825477,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1457.0,down,LOW,keyword article,informational,91-180,91-180,1000-2000,good,page_1
2,3,content_d6570c51c9bd,client_3fdba35f04,81.430346,random_forest,0.847372,0.695884,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1362.0,down,MEDIUM,keyword article,informational,91-180,91-180,1000-2000,moderate,striking
3,4,content_72e800a9c214,client_3fdba35f04,81.034960,random_forest,0.774371,0.842545,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1371.0,down,MEDIUM,keyword article,commercial,91-180,91-180,1000-2000,good,page_1
4,5,content_e04eb9549989,client_3fdba35f04,80.873188,random_forest,0.814805,0.749468,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1408.0,down,LOW,keyword article,informational,91-180,91-180,1000-2000,good,page_1


Why ML: A fixed rule like sorting strictly by "lowest price" fails because users trade off price against duration, departure times, brand loyalty, and stops. ML learns complex, non-linear feature interactions that simple if/else conditions cannot capture.

In [11]:
if 'price' in df.columns and target_column in df.columns:
    correlation = df[['price', target_column]].corr().iloc[0, 1]
    print(f"Correlation between price alone and target: {correlation:.3f}")
    print("Conclusion: Weak single-feature correlation proves a simple rule is insufficient.")